# AnimalEmotion AI – Deep Learning-Based Animal Emotion Recognition System
## Phase 1: Dataset Audit

Evidence-driven dataset auditing only. This notebook does not train models, create data splits, or modify/delete original images. Dataset page: https://www.kaggle.com/datasets/kk2004/animalemotion/data

Labels are not inferred as authoritative from folder names or suggestive column names. Human verification is required.

## Upload instructions (manual ZIP workflow)
1. Download the dataset ZIP manually from https://www.kaggle.com/datasets/kk2004/animalemotion/data (sign in to Kaggle and accept any required terms).
2. Run the upload cell and select exactly one `.zip` file. No Kaggle API credentials, KaggleHub, Colab Secrets, CLI, or automatic download are used.
3. The ZIP is retained in `/content/animalemotion/uploads/`. Validated contents are safely extracted to `/content/animalemotion/dataset/`; reports remain under `/content/animalemotion/reports/`. Existing ZIPs and extracted files are never automatically deleted or overwritten. If a prior run left colliding extracted files, use a fresh Colab runtime or manually choose an empty dataset directory.
4. Run all cells top-to-bottom. Review annotation mappings and license/usage questions before any future modeling.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, hashlib, stat, shutil, random, re, subprocess, datetime, platform, ast
import numpy as np, pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError, ImageFile
from tqdm.auto import tqdm
import matplotlib.pyplot as plt, seaborn as sns
import torch, torchvision, cv2, sklearn
try:
    import psutil
except Exception:
    psutil = None
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False; sns.set_theme(style='whitegrid')
PROJECT_DIR=Path('/content/animalemotion'); UPLOAD_DIR=PROJECT_DIR/'uploads'; DATA_DIR=Path('/content/animalemotion/dataset'); REPORT_DIR=Path('/content/animalemotion/reports')
for directory in (UPLOAD_DIR, DATA_DIR, REPORT_DIR): directory.mkdir(parents=True,exist_ok=True)
print('Python',platform.python_version(),'| PyTorch',torch.__version__,'| torchvision',torchvision.__version__,'| OpenCV',cv2.__version__)
print('CUDA available:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
def resource_snapshot(stage):
    row={'stage':stage,'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'rss_bytes':None,'available_memory_bytes':None}
    if psutil:
        try: row.update(rss_bytes=psutil.Process().memory_info().rss,available_memory_bytes=psutil.virtual_memory().available)
        except Exception as exc: row['error']=str(exc)
    return row
resource_log=[resource_snapshot('setup')]
DATASET_SOURCE_URL='https://www.kaggle.com/datasets/kk2004/animalemotion/data'
DATASET_SLUG='kk2004/animalemotion'

## Manual ZIP upload, validation, and safe extraction
The whole archive is preflighted before writing: absolute paths, `..`, symlinks, duplicate member paths, and destination collisions are rejected. Extraction writes files individually and exclusively (`xb`), so existing data is not overwritten. Invalid ZIPs and partial extraction failures are reported with actionable errors; partial files are retained rather than silently deleted.

In [ ]:
from google.colab import files
print('Please upload exactly one AnimalEmotion dataset ZIP downloaded from the Kaggle dataset page.')
uploaded=files.upload()
if not uploaded: raise RuntimeError('No file uploaded. Run this cell again and select one dataset ZIP.')
zip_names=[name for name in uploaded if str(name).lower().endswith('.zip')]
if len(uploaded)!=1 or len(zip_names)!=1: raise ValueError(f'Expected exactly one uploaded ZIP; received {len(uploaded)} file(s), {len(zip_names)} ZIP(s).')
original_name=Path(zip_names[0]).name
if not original_name: raise ValueError('Uploaded filename is empty or invalid.')
archive_bytes=uploaded[zip_names[0]]
if not archive_bytes: raise ValueError('Uploaded ZIP is empty.')
archive_digest=hashlib.sha256(archive_bytes).hexdigest()
zip_path=UPLOAD_DIR/original_name
if zip_path.exists():
    if file_hash := hashlib.sha256(zip_path.read_bytes()).hexdigest() == archive_digest:
        print('A byte-identical ZIP already exists; keeping it and using the uploaded copy.')
    else:
        stem,suffix=zip_path.stem,zip_path.suffix; i=1
        while zip_path.exists(): zip_path=UPLOAD_DIR/f'{stem}_{i}{suffix}'; i+=1
with open(zip_path,'xb') if not zip_path.exists() else open(UPLOAD_DIR/f'{zip_path.stem}_upload_{archive_digest[:8]}{zip_path.suffix}','xb') as stream:
    stream.write(archive_bytes)
    zip_path=Path(stream.name)
print('Uploaded ZIP saved:',zip_path.name,'| size MB:',round(zip_path.stat().st_size/1024**2,2))
if not zipfile.is_zipfile(zip_path): raise ValueError('The uploaded file has a .zip extension but is not a valid ZIP archive.')
def preflight_and_extract(archive_path,destination):
    root=destination.resolve(); entries=[]; names=set(); rejected=[]
    with zipfile.ZipFile(archive_path,'r') as archive:
        bad=archive.testzip()
        if bad is not None: raise ValueError(f'ZIP CRC check failed for member: {bad}')
        for member in archive.infolist():
            name=member.filename; rel=Path(name); mode=member.external_attr>>16
            if name in names: raise ValueError(f'Duplicate ZIP member name: {name}')
            names.add(name)
            if name.startswith('/') or re.match(r'^[A-Za-z]:',name) or '..' in rel.parts or stat.S_ISLNK(mode):
                rejected.append({'entry':name,'reason':'absolute/traversal path or symbolic link'}); continue
            target=(root/rel).resolve()
            if target!=root and root not in target.parents: rejected.append({'entry':name,'reason':'resolved outside dataset directory'}); continue
            if target.exists() and not member.is_dir(): raise FileExistsError(f'Extraction would overwrite existing data: {target}. Use a fresh Colab runtime or an empty dataset folder; nothing was overwritten.')
            entries.append((member,target))
        if rejected: raise ValueError('Unsafe ZIP entries rejected before extraction: '+json.dumps(rejected[:20]))
        written=[]
        try:
            for member,target in entries:
                if member.is_dir(): target.mkdir(parents=True,exist_ok=True); continue
                target.parent.mkdir(parents=True,exist_ok=True)
                with archive.open(member,'r') as src, open(target,'xb') as dst: shutil.copyfileobj(src,dst)
                written.append(target)
        except Exception as exc:
            raise RuntimeError(f'Extraction stopped at {target}; {len(written)} earlier files remain in place and were not deleted. Original ZIP retained. Cause: {type(exc).__name__}: {exc}') from exc
    return written
try:
    extracted_files=preflight_and_extract(zip_path,DATA_DIR)
except zipfile.BadZipFile as exc:
    raise ValueError(f'Invalid or damaged ZIP archive: {exc}') from exc
if not extracted_files and not any(DATA_DIR.rglob('*')): raise RuntimeError('ZIP extraction produced no files. Confirm the selected archive contains the dataset.')
if not any(DATA_DIR.rglob('*')): raise RuntimeError('Extraction verification failed: dataset directory is empty.')
upload_metadata={'dataset_slug':DATASET_SLUG,'source_url':DATASET_SOURCE_URL,'uploaded_filename':zip_path.name,'archive_sha256':archive_digest,'archive_bytes':zip_path.stat().st_size,'upload_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'dataset_version':'unknown; manual download version not embedded/verified','license_metadata':'not retrieved via API; inspect Kaggle dataset page and included license files','usage_review':{'reviewed':False,'status':'manual review required'}}
(REPORT_DIR/'dataset_metadata_and_license_review.json').write_text(json.dumps(upload_metadata,indent=2))
(REPORT_DIR/'safe_extraction.json').write_text(json.dumps({'archive':zip_path.name,'extracted_file_count':len(extracted_files),'extraction_root':str(DATA_DIR),'rejected_entries':[],'verification_file_count':sum(1 for p in DATA_DIR.rglob('*') if p.is_file())},indent=2))
resource_log.append(resource_snapshot('after_extraction'))
print('Extraction verified:',len(extracted_files),'new files | dataset directory:',DATA_DIR)

## Recursive inspection, image discovery, integrity, and EXIF-aware dimensions

In [ ]:
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.gif','.tif','.tiff','.webp','.jfif','.heic','.heif'}; ANNOT_EXTS={'.csv','.json','.xlsx','.xls','.yaml','.yml','.xml','.txt'}
all_files=[p for p in DATA_DIR.rglob('*') if p.is_file() and UPLOAD_DIR not in p.parents and REPORT_DIR not in p.parents]
image_files=[p for p in all_files if p.suffix.lower() in IMAGE_EXTS]; annotation_files=[p for p in all_files if p.suffix.lower() in ANNOT_EXTS]
folder_counts={}
for path in all_files:
    rel=path.relative_to(DATA_DIR)
    for depth in range(1,len(rel.parts)):
        key='/'.join(rel.parts[:depth]); folder_counts[key]=folder_counts.get(key,0)+1
folder_df=pd.DataFrame([{'folder':key,'file_count':count} for key,count in sorted(folder_counts.items())],columns=['folder','file_count'])
if folder_df.empty: folder_df=pd.DataFrame({'folder':pd.Series(dtype='string'),'file_count':pd.Series(dtype='int64')})
folder_df.to_csv(REPORT_DIR/'folder_hierarchy.csv',index=False)
IMAGE_COLUMNS=['relative_path','filename','extension','status','format','mode','channels','width','height','aspect_ratio','file_bytes','sha256','has_transparency','error']
def file_sha256(path):
    digest=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
def inspect_image(path):
    row={'relative_path':path.relative_to(DATA_DIR).as_posix(),'filename':path.name,'extension':path.suffix.lower(),'status':'unreadable','format':None,'mode':None,'channels':None,'width':None,'height':None,'aspect_ratio':None,'file_bytes':path.stat().st_size,'sha256':None,'has_transparency':None,'error':''}
    try: row['sha256']=file_sha256(path)
    except Exception as exc: row['error']=f'hash failed: {type(exc).__name__}: {exc}'; return row
    try:
        with Image.open(path) as raw: raw.verify()
        with Image.open(path) as raw:
            raw.load(); oriented=ImageOps.exif_transpose(raw); width,height=oriented.size
            row.update(status='ok',format=raw.format,mode=oriented.mode,channels=len(oriented.getbands()),width=width,height=height,aspect_ratio=width/height if height else None,has_transparency=('A' in oriented.getbands() or 'transparency' in raw.info))
    except Exception as exc:
        if path.suffix.lower() in {'.heic','.heif'}: status='unsupported_format'
        elif isinstance(exc,OSError): status='corrupted'
        else: status='unreadable'
        row.update(status=status,error=f'{type(exc).__name__}: {exc}')
    return row
resource_log.append(resource_snapshot('before_image_audit')); rows=[]
for idx,path in enumerate(tqdm(image_files,desc='Image integrity audit')):
    try: rows.append(inspect_image(path))
    except Exception as exc:
        rows.append({'relative_path':path.relative_to(DATA_DIR).as_posix(),'filename':path.name,'extension':path.suffix.lower(),'status':'unreadable','format':None,'mode':None,'channels':None,'width':None,'height':None,'aspect_ratio':None,'file_bytes':path.stat().st_size,'sha256':None,'has_transparency':None,'error':f'{type(exc).__name__}: {exc}'})
    if (idx+1)%250==0: resource_log.append(resource_snapshot(f'image_audit_{idx+1}'))
inventory=pd.DataFrame(rows,columns=IMAGE_COLUMNS)
if inventory.empty:
    inventory=pd.DataFrame({col:pd.Series(dtype=('string' if col in {'relative_path','filename','extension','status','format','mode','sha256','error'} else 'object')) for col in IMAGE_COLUMNS})
inventory.to_csv(REPORT_DIR/'image_inventory.csv',index=False)
valid=inventory[inventory.status=='ok'].copy(); corrupt=inventory[inventory.status=='corrupted'].copy(); unsupported=inventory[inventory.status=='unsupported_format'].copy(); unreadable=inventory[inventory.status=='unreadable'].copy()
corrupt.to_csv(REPORT_DIR/'corrupted_images.csv',index=False); unsupported.to_csv(REPORT_DIR/'unsupported_images.csv',index=False); unreadable.to_csv(REPORT_DIR/'unreadable_images.csv',index=False)
print('Files:',len(all_files),'| image candidates:',len(image_files),'| readable:',len(valid),'| corrupted:',len(corrupt),'| unsupported:',len(unsupported),'| unreadable:',len(unreadable))

## Image properties, duplicate detection, optional scalable near-duplicates, and visual review
EXIF orientation is applied to displayed images and reported dimensions only; source image bytes are unchanged. Perceptual hashing uses banded candidate generation with bucket and pair limits, avoiding unrestricted all-pairs comparisons. It remains optional and may miss near-duplicates.

In [ ]:
if len(valid):
    dimensions={'readable_images':int(len(valid)),'width_min':int(valid.width.min()),'width_median':float(valid.width.median()),'width_max':int(valid.width.max()),'height_min':int(valid.height.min()),'height_median':float(valid.height.median()),'height_max':int(valid.height.max()),'aspect_ratio_median':float(valid.aspect_ratio.median()),'formats':valid['format'].value_counts().to_dict(),'modes':valid['mode'].value_counts().to_dict(),'transparent_count':int(valid.has_transparency.sum()),'file_bytes_min':int(valid.file_bytes.min()),'file_bytes_median':float(valid.file_bytes.median()),'file_bytes_max':int(valid.file_bytes.max())}
else: dimensions={'readable_images':0}
(REPORT_DIR/'dimension_summary.json').write_text(json.dumps(dimensions,indent=2))
if len(valid):
    counts=valid.groupby('sha256',dropna=True).size(); duplicate_hashes=counts[counts>1].index
    exact_dups=valid[valid.sha256.isin(duplicate_hashes)].sort_values(['sha256','relative_path']).copy()
else: exact_dups=pd.DataFrame(columns=IMAGE_COLUMNS)
exact_dups.to_csv(REPORT_DIR/'exact_duplicates.csv',index=False)
exact_groups=pd.DataFrame([{'sha256':key,'paths':' | '.join(group.relative_path.astype(str)),'count':int(len(group))} for key,group in exact_dups.groupby('sha256')],columns=['sha256','paths','count'])
exact_groups.to_csv(REPORT_DIR/'exact_duplicate_groups.csv',index=False)
ENABLE_PHASH=False; PHASH_THRESHOLD=6; PHASH_MAX_BUCKET=1000; PHASH_MAX_PAIRS=200000
phash_rows=[]; near_pairs=[]; phash_skipped=[]
if ENABLE_PHASH and len(valid):
    subprocess.run([sys.executable,'-m','pip','install','-q','ImageHash'],check=True); import imagehash
    for rel in tqdm(valid.relative_path.astype(str),desc='Perceptual hashes'):
        try:
            with Image.open(DATA_DIR/rel) as raw: phash_rows.append({'relative_path':rel,'phash':str(imagehash.phash(ImageOps.exif_transpose(raw).convert('RGB'),hash_size=8))})
        except Exception as exc: phash_skipped.append({'path':rel,'error':str(exc)})
    buckets={}
    for index,row in enumerate(phash_rows):
        bits=f'{int(row["phash"],16):064b}'
        for band in range(8): buckets.setdefault((band,bits[band*8:(band+1)*8]),[]).append(index)
    candidate_pairs=set()
    for key,indices in buckets.items():
        if len(indices)>PHASH_MAX_BUCKET: phash_skipped.append({'bucket':str(key),'reason':'bucket cap exceeded','size':len(indices)}); continue
        for pos,a in enumerate(indices):
            for b in indices[pos+1:]:
                candidate_pairs.add((min(a,b),max(a,b)))
                if len(candidate_pairs)>=PHASH_MAX_PAIRS: break
            if len(candidate_pairs)>=PHASH_MAX_PAIRS: break
        if len(candidate_pairs)>=PHASH_MAX_PAIRS: break
    for a,b in candidate_pairs:
        distance=imagehash.hex_to_hash(phash_rows[a]['phash'])-imagehash.hex_to_hash(phash_rows[b]['phash'])
        if distance<=PHASH_THRESHOLD: near_pairs.append({'path_a':phash_rows[a]['relative_path'],'path_b':phash_rows[b]['relative_path'],'hamming_distance':int(distance),'threshold':PHASH_THRESHOLD})
pd.DataFrame(phash_rows,columns=['relative_path','phash']).to_csv(REPORT_DIR/'perceptual_hashes.csv',index=False)
pd.DataFrame(near_pairs,columns=['path_a','path_b','hamming_distance','threshold']).to_csv(REPORT_DIR/'perceptual_duplicate_candidates.csv',index=False)
(REPORT_DIR/'perceptual_hash_skipped.json').write_text(json.dumps(phash_skipped,indent=2))
if len(valid):
    fig,axes=plt.subplots(2,2,figsize=(13,9)); axes[0,0].hist(valid.width,bins=40); axes[0,0].set_title('Width (px)'); axes[0,1].hist(valid.height,bins=40); axes[0,1].set_title('Height (px)'); axes[1,0].scatter(valid.width,valid.height,alpha=.4); axes[1,0].set_title('Resolution'); axes[1,1].hist(valid.file_bytes/1024,bins=40); axes[1,1].set_title('File size (KB)'); plt.tight_layout(); plt.savefig(REPORT_DIR/'image_property_plots.png',dpi=150); plt.show()
def show_gallery(paths,title,filename):
    paths=list(paths)[:20]
    if not paths: print(title+': no examples available'); return
    columns=5; fig,axes=plt.subplots((len(paths)+columns-1)//columns,columns,figsize=(15,3*((len(paths)+columns-1)//columns))); axes=np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis('off')
    for ax,rel in zip(axes,paths):
        try:
            with Image.open(DATA_DIR/rel) as raw: ax.imshow(ImageOps.exif_transpose(raw).convert('RGB')); ax.set_title(Path(rel).name[:25],fontsize=8)
        except Exception as exc: ax.text(.5,.5,f'Unavailable: {type(exc).__name__}',ha='center')
    fig.suptitle(title); plt.tight_layout(); plt.savefig(REPORT_DIR/filename,dpi=130); plt.show()
rng=np.random.default_rng(SEED); sample=valid.relative_path.astype(str).tolist(); rng.shuffle(sample); show_gallery(sample,'Readable image examples','sample_images.png')
show_gallery(exact_dups.relative_path.drop_duplicates().astype(str).tolist() if len(exact_dups) else [],'Exact duplicate examples','duplicate_examples.png')

## Annotation discovery and image mapping
Filename/path columns are discovered heuristically and normalized only as safe relative keys. Annotation labels remain candidates, not authoritative labels. Reports identify unmatched annotation rows, images without a match, duplicate references, and conflicting candidate values.

In [ ]:
annotation_files=[p for p in all_files if p.suffix.lower() in ANNOT_EXTS]; annotation_manifest=[]; annotation_tables={}
for path in annotation_files:
    rel=path.relative_to(DATA_DIR).as_posix(); record={'relative_path':rel,'suffix':path.suffix.lower(),'status':'unreadable','row_count':None,'columns':[],'error':''}
    try:
        if path.suffix.lower()=='.csv': table=pd.read_csv(path); annotation_tables[rel]=table
        elif path.suffix.lower()=='.json':
            obj=json.loads(path.read_text(encoding='utf-8')); table=pd.DataFrame(obj) if isinstance(obj,list) and (not obj or isinstance(obj[0],dict)) else None
            if table is None: record.update(status='readable_non_tabular_json',row_count=len(obj) if isinstance(obj,list) else None); annotation_manifest.append(record); continue
            annotation_tables[rel]=table
        elif path.suffix.lower() in {'.xlsx','.xls'}:
            table=pd.read_excel(path); annotation_tables[rel]=table
        else: record.update(status='present_not_parsed'); annotation_manifest.append(record); continue
        record.update(status='readable',row_count=len(table),columns=[str(c) for c in table.columns])
    except Exception as exc: record['error']=f'{type(exc).__name__}: {exc}'
    annotation_manifest.append(record)
ann_manifest=pd.DataFrame(annotation_manifest,columns=['relative_path','suffix','status','row_count','columns','error'])
if ann_manifest.empty: ann_manifest=pd.DataFrame({'relative_path':pd.Series(dtype='string'),'suffix':pd.Series(dtype='string'),'status':pd.Series(dtype='string'),'row_count':pd.Series(dtype='object'),'columns':pd.Series(dtype='object'),'error':pd.Series(dtype='string')})
ann_manifest.to_csv(REPORT_DIR/'annotation_manifest.csv',index=False)
def norm_name(value): return re.sub(r'[^a-z0-9]+','_',str(value).lower()).strip('_')
def safe_key(value):
    text=str(value).strip().replace('\\','/')
    if text.startswith('/') or re.match(r'^[a-zA-Z]:/',text) or '..' in Path(text).parts: return None
    while text.startswith('./'): text=text[2:]
    return text.lower()
image_key_index={}
for rel in inventory.relative_path.astype(str):
    for key in {safe_key(rel),safe_key(Path(rel).name)}:
        if key: image_key_index.setdefault(key,[]).append(rel)
path_tokens=('file','filename','filepath','path','image','img','photo','picture'); label_tokens={'emotion','emotions','expression','mood','sentiment','species','animal','animal_type','animal_class','breed','label','class','category','target'}
map_rows=[]; label_candidates=[]; reference_rows=[]
for source,table in annotation_tables.items():
    path_cols=[]; label_cols=[]
    for col in table.columns:
        name=norm_name(col)
        if any(token in name for token in path_tokens): path_cols.append(col)
        if name in label_tokens or any(token in name for token in ('emotion','species','animal_type')): label_cols.append(col)
    for col in table.columns:
        if col not in path_cols:
            values=table[col].dropna().astype(str).head(100)
            if len(values) and sum(Path(v.replace('\\','/')).suffix.lower() in IMAGE_EXTS for v in values)/len(values)>=0.5: path_cols.append(col)
    for col in label_cols: label_candidates.append({'source':source,'column':str(col),'candidate_only_not_authoritative':True,'unique_values':int(table[col].nunique(dropna=True))})
    if not path_cols: map_rows.append({'source':source,'row_index':None,'annotation_reference':None,'status':'no_path_column','match_count':0,'matched_paths':'','candidate_labels_json':'{}'}); continue
    for row_index,row in table.iterrows():
        refs=[]; original=None
        for col in path_cols:
            if pd.notna(row[col]):
                original=str(row[col]); key=safe_key(original); basename=safe_key(Path(original.replace('\\','/')).name)
                refs.extend(image_key_index.get(key,[]) if key else []); refs.extend(image_key_index.get(basename,[]) if basename else [])
        matches=sorted(set(refs)); labels={str(col):str(row[col]) for col in label_cols if pd.notna(row[col])}
        map_rows.append({'source':source,'row_index':int(row_index),'annotation_reference':original,'status':'matched' if matches else 'unmatched_annotation','match_count':len(matches),'matched_paths':'|'.join(matches),'candidate_labels_json':json.dumps(labels,ensure_ascii=False)})
        for image_path in matches: reference_rows.append({'image_path':image_path,'source':source,'row_index':int(row_index),'candidate_labels_json':json.dumps(labels,ensure_ascii=False)})
mapping=pd.DataFrame(map_rows,columns=['source','row_index','annotation_reference','status','match_count','matched_paths','candidate_labels_json'])
if mapping.empty: mapping=pd.DataFrame(columns=['source','row_index','annotation_reference','status','match_count','matched_paths','candidate_labels_json'])
mapping.to_csv(REPORT_DIR/'annotation_image_mapping.csv',index=False)
mapping[mapping.status=='unmatched_annotation'].to_csv(REPORT_DIR/'unmatched_annotations.csv',index=False)
refs=pd.DataFrame(reference_rows,columns=['image_path','source','row_index','candidate_labels_json'])
if refs.empty: refs=pd.DataFrame(columns=['image_path','source','row_index','candidate_labels_json'])
refs.to_csv(REPORT_DIR/'annotation_references.csv',index=False)
duplicate_refs=refs[refs.duplicated('image_path',keep=False)].sort_values('image_path') if len(refs) else refs.copy(); duplicate_refs.to_csv(REPORT_DIR/'duplicate_annotation_references.csv',index=False)
conflicts=[]
for image_path,group in refs.groupby('image_path') if len(refs) else []:
    values={}
    for payload in group.candidate_labels_json:
        for col,val in json.loads(payload).items(): values.setdefault(col,set()).add(val)
    for col,vals in values.items():
        if len(vals)>1: conflicts.append({'image_path':image_path,'column':col,'values':' | '.join(sorted(vals)),'reference_count':len(group)})
pd.DataFrame(conflicts,columns=['image_path','column','values','reference_count']).to_csv(REPORT_DIR/'conflicting_labels.csv',index=False)
matched_images=set(refs.image_path.astype(str)) if len(refs) else set(); unlabeled=inventory[~inventory.relative_path.astype(str).isin(matched_images)].copy() if len(inventory) else inventory.copy(); unlabeled.to_csv(REPORT_DIR/'unlabeled_images.csv',index=False)
pd.DataFrame(label_candidates,columns=['source','column','candidate_only_not_authoritative','unique_values']).to_csv(REPORT_DIR/'label_column_candidates.csv',index=False)
print('Unmatched annotations:',int((mapping.status=='unmatched_annotation').sum()) if len(mapping) else 0,'| unlabeled images:',len(unlabeled),'| duplicate refs:',len(duplicate_refs),'| conflicting label records:',len(conflicts))

## Explicitly verified labels, class distribution, and galleries
No label source is considered verified by default. After checking documentation and mapping results, explicitly set `VERIFIED_LABEL_COLUMNS = {'relative/annotations.csv': 'exact_column_name'}` in the next cell. The explicit choice means analyst review, not automatic validation of annotation correctness. Only unique image matches are used for distributions and per-class galleries.

In [ ]:
VERIFIED_LABEL_COLUMNS={}  # analyst review required: {annotation_relative_path: exact_label_column}
verified_rows=[]
for source,column in VERIFIED_LABEL_COLUMNS.items():
    if source not in annotation_tables: print('Configured annotation source is not a readable table:',source); continue
    table=annotation_tables[source]
    if column not in table.columns: print('Configured column missing:',source,column); continue
    selected=mapping[(mapping.source==source)&(mapping.match_count==1)&(mapping.status=='matched')] if len(mapping) else mapping
    for _,record in selected.iterrows():
        row_index=int(record.row_index); label=table.iloc[row_index][column]
        if pd.notna(label): verified_rows.append({'source':source,'image_path':record.matched_paths,'label_column':column,'label':str(label)})
verified=pd.DataFrame(verified_rows,columns=['source','image_path','label_column','label'])
class_dist=verified.groupby(['source','label_column','label']).size().reset_index(name='count') if len(verified) else pd.DataFrame(columns=['source','label_column','label','count'])
class_dist.to_csv(REPORT_DIR/'class_distribution.csv',index=False); verified.to_csv(REPORT_DIR/'verified_label_image_mapping.csv',index=False)
imbalance=[]
for (source,column),group in class_dist.groupby(['source','label_column']) if len(class_dist) else []:
    counts=group['count']; imbalance.append({'source':source,'column':column,'class_count':len(counts),'total_labeled_images':int(counts.sum()),'min_class':int(counts.min()),'max_class':int(counts.max()),'max_min_ratio':float(counts.max()/counts.min()) if counts.min() else None})
pd.DataFrame(imbalance,columns=['source','column','class_count','total_labeled_images','min_class','max_class','max_min_ratio']).to_csv(REPORT_DIR/'imbalance_analysis.csv',index=False)
if len(class_dist):
    plt.figure(figsize=(12,max(5,min(16,len(class_dist)/2)))); sns.barplot(data=class_dist,x='count',y='label',hue='label_column',dodge=False); plt.tight_layout(); plt.savefig(REPORT_DIR/'class_distribution.png',dpi=150); plt.show()
for (source,column,label),group in verified.groupby(['source','label_column','label']) if len(verified) else []:
    slug=re.sub(r'[^A-Za-z0-9_-]+','_',str(label))[:45]; show_gallery(group.image_path.astype(str).tolist(),f'Verified {column}: {label}',f'class_gallery_{slug}.png')
print('Explicit analyst-configured image-label rows:',len(verified))

## Suspicious records, leakage assessment, dataset metadata, and verdict
Repeated subject identifiers are possible overlap indicators only. A matching filename stem is not proof of duplicate content. The Kaggle dataset version and license are not requested via API: metadata records the uploaded archive hash and download/upload date, with version/usage explicitly requiring manual review.

In [ ]:
stem_counts=valid.assign(stem=valid.filename.map(lambda name:Path(name).stem)).groupby('stem').size() if len(valid) else pd.Series(dtype=int); repeated_stems=stem_counts[stem_counts>1]
subject_records=[]
for source,table in annotation_tables.items():
    for column in table.columns:
        if any(token in norm_name(column) for token in ('subject','individual','animal_id','patient','identity')):
            counts=table[column].dropna().astype(str).value_counts()
            for subject,count in counts[counts>1].items(): subject_records.append({'source':source,'column':str(column),'subject_identifier':subject,'rows':int(count),'interpretation':'possible overlap; identity semantics unverified'})
subject_df=pd.DataFrame(subject_records,columns=['source','column','subject_identifier','rows','interpretation']); subject_df.to_csv(REPORT_DIR/'possible_subject_overlap.csv',index=False)
suspicious=[]
for _,row in class_dist[class_dist['count']<5].iterrows(): suspicious.append({'issue':'rare_verified_class','source':row.source,'column':row.label_column,'label':row.label,'count':int(row['count'])})
for stem,count in repeated_stems.items(): suspicious.append({'issue':'repeated_filename_stem','stem':stem,'count':int(count)})
if len(valid):
    for _,row in valid[(valid.width<32)|(valid.height<32)|(valid.aspect_ratio<.25)|(valid.aspect_ratio>4)].iterrows(): suspicious.append({'issue':'unusual_dimensions_or_aspect','path':row.relative_path,'width':row.width,'height':row.height,'aspect_ratio':row.aspect_ratio})
pd.DataFrame(suspicious,columns=['issue','source','column','label','count','stem','path','width','height','aspect_ratio']).to_csv(REPORT_DIR/'suspicious_review.csv',index=False)
leakage={'exact_duplicate_groups':int(len(exact_groups)),'files_in_exact_duplicate_groups':int(len(exact_dups)),'near_duplicate_candidates':len(near_pairs),'near_duplicate_enabled':ENABLE_PHASH,'repeated_filename_stems':int(len(repeated_stems)),'possible_subject_overlap_records':len(subject_df),'unmatched_annotation_rows':int((mapping.status=='unmatched_annotation').sum()) if len(mapping) else 0,'unlabeled_images':len(unlabeled),'duplicate_annotation_references':len(duplicate_refs),'conflicting_label_records':len(conflicts),'split_created':False,'limitations':['Subject-column matching is heuristic, identity is unverified','Perceptual blocking is capped and may miss matches','Filename stem overlap does not establish duplicate images','No split/group assignment or training was performed']}
(REPORT_DIR/'leakage_assessment.json').write_text(json.dumps(leakage,indent=2))
issues=[]
for label,frame in [('corrupted',corrupt),('unsupported',unsupported),('unreadable',unreadable)]:
    if len(frame): issues.append(f'{len(frame)} {label} image candidates')
if len(exact_groups): issues.append(f'{len(exact_groups)} exact duplicate groups')
if not len(verified): issues.append('No analyst-verified label assignments configured')
suitability='potentially suitable after annotation, duplicate, and license review' if len(verified) and not len(corrupt) else 'not established for supervised training; review label evidence and audit findings first'
verdict={'phase':1,'dataset_suitability':suitability,'file_counts':{'all_files':len(all_files),'image_candidates':len(image_files),'readable':len(valid),'corrupted':len(corrupt),'unsupported':len(unsupported),'unreadable':len(unreadable),'annotation_candidates':len(annotation_files)},'detected_issues':issues,'unresolved_limitations':leakage['limitations']+['Kaggle version/usage terms require manual verification'],'phase_2_recommendations':['Manually validate mapping and label ontology','Review unmatched, duplicate-reference, conflicting-label, and unlabeled records','Verify dataset license and Kaggle terms','Resolve exact/near duplicates and possible subject overlap before designing a split']}
(REPORT_DIR/'phase1_verdict.json').write_text(json.dumps(verdict,indent=2))
(REPORT_DIR/'audit_summary.json').write_text(json.dumps({'project':'AnimalEmotion AI','dataset_slug':DATASET_SLUG,'seed':SEED,'verdict':verdict,'leakage':leakage,'dimensions':dimensions,'verified_label_rows':len(verified)},indent=2))
resource_log.append(resource_snapshot('audit_complete')); pd.DataFrame(resource_log).to_csv(REPORT_DIR/'resource_monitoring.csv',index=False)
print(json.dumps(verdict,indent=2))

## Report validation and download
Expected report artifacts are checked, then the manifest is written after all analysis reports. The ZIP and extracted images remain on disk; the notebook does not delete source data.

In [ ]:
EXPECTED_REPORTS=['dataset_metadata_and_license_review.json','safe_extraction.json','folder_hierarchy.csv','image_inventory.csv','corrupted_images.csv','unsupported_images.csv','unreadable_images.csv','dimension_summary.json','exact_duplicates.csv','exact_duplicate_groups.csv','perceptual_hashes.csv','perceptual_duplicate_candidates.csv','perceptual_hash_skipped.json','annotation_manifest.csv','annotation_image_mapping.csv','unmatched_annotations.csv','annotation_references.csv','duplicate_annotation_references.csv','conflicting_labels.csv','unlabeled_images.csv','label_column_candidates.csv','class_distribution.csv','verified_label_image_mapping.csv','imbalance_analysis.csv','leakage_assessment.json','possible_subject_overlap.csv','suspicious_review.csv','phase1_verdict.json','audit_summary.json','resource_monitoring.csv']
missing=[name for name in EXPECTED_REPORTS if not (REPORT_DIR/name).is_file()]
if missing: raise RuntimeError('Expected reports missing: '+', '.join(missing))
manifest={'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'report_directory':str(REPORT_DIR),'expected_reports':EXPECTED_REPORTS,'report_files':sorted(path.name for path in REPORT_DIR.iterdir() if path.is_file()),'archive_name':zip_path.name,'archive_sha256':archive_digest}
(REPORT_DIR/'REPORT_MANIFEST.json').write_text(json.dumps(manifest,indent=2))
reloaded=json.loads((REPORT_DIR/'REPORT_MANIFEST.json').read_text())
if set(EXPECTED_REPORTS)-set(reloaded['report_files']): raise RuntimeError('Manifest validation failed to include all expected reports.')
bundle=PROJECT_DIR/'animalemotion_phase1_audit_reports.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(REPORT_DIR.iterdir()):
        if path.is_file(): archive.write(path,arcname='reports/'+path.name)
print('Reports validated:',len(EXPECTED_REPORTS),'| report folder:',REPORT_DIR,'| bundle:',bundle)
try:
    files.download(str(bundle))
except Exception as exc: print('Automatic browser download unavailable:',exc)
print('Notebook finished: no training, data split, or source-image modification.')

## Static notebook validation (does not execute data-dependent cells)
This final check validates cell IDs, Python syntax, absence of credential/automatic-download code patterns, dataset/report path consistency, and required report names. Actual ZIP upload/extraction and dataset-dependent outputs can only be verified by running in Colab.

In [ ]:
import nbformat
notebook_path=Path('/content/AnimalEmotion_AI_Phase1_Audit.ipynb')
if notebook_path.exists():
    nb=nbformat.read(notebook_path,as_version=4)
else:
    print('Notebook file not mounted locally; static checks on current in-memory cell sources are limited.')
ids=[cell.get('id') for cell in globals().get('nb',{}).get('cells',[])] if notebook_path.exists() else []
if ids and (len(ids)!=len(set(ids)) or any(not item for item in ids)): raise AssertionError('Notebook cell IDs are missing or duplicated.')
if notebook_path.exists():
    nbformat.validate(nb)
    for index,cell in enumerate(nb.cells):
        if cell.cell_type=='code': ast.parse(cell.source,filename=f'cell_{index}')
    text='\n'.join(cell.source for cell in nb.cells)
    forbidden=['kagglehub','KAGGLE_KEY','KAGGLE_USERNAME','kaggle.json','kaggle datasets download','userdata.get']
    found=[term for term in forbidden if term.lower() in text.lower()]
    if found: raise AssertionError('Forbidden Kaggle authentication/download pattern found: '+str(found))
    assert "Path('/content/animalemotion/dataset')" in text
    assert "Path('/content/animalemotion/reports')" in text
    print('Notebook JSON, cell IDs, Python syntax, credential removal, paths, and report expectations validated.')
else: print('Run nbformat/AST validation after opening the notebook in Colab or save the notebook to /content first.')